# 03 · Galería de visualizaciones de Work-Olist

Más de veinte representaciones: distribuciones, comparaciones, tiempo, asociaciones, matrices, mapas y diagnóstico visual, explicando cuándo conviene cada una y qué puede sesgar su interpretación.

**Modalidad:** práctica guiada, código ejecutable, interpretación y ejercicios. **Datos:** nueve CSV aportados por el usuario en `data/raw/`, con el esquema relacional de Olist. Se calculan resultados sobre estos archivos, sin simular registros ni cambiar las fuentes.

## Objetivos, criterio visual y guía de uso
El objetivo no es hacer todos los gráficos por rutina. Cada sección formula una **pregunta analítica** y propone el tipo de representación adecuado. Se revisan Gestalt, posición frente a área, escalas, paletas, superposición, tamaño de muestra, etiquetas, exportación y lectura crítica, como indican las notas del curso anterior.

Los gráficos se construyen con `matplotlib` y, en ejemplos puntuales, `pandas` o SciPy. Los **nueve CSV recibidos** están incluidos en `data/raw`; no se sustituyen por datos simulados. Cada gráfico utiliza datos y cálculos derivados, **no ilustraciones generadas**.

In [ ]:
from pathlib import Path
import sys
from IPython.display import display
import numpy as np
import pandas as pd
# Detectar carpeta del proyecto desde raíz/ o notebooks/ (sin rutas absolutas).
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "src" / "olist_tools.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Ejecute el notebook dentro de la carpeta Work_Olist_Notebooks_2026 (README.md).")
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from src.olist_tools import (read_tables, build_order_level, build_item_level,
                            table_inventory, missing_profile, safe_minmax, safe_zscore)
TABLAS, ORIGEN = read_tables(ROOT)
print("ORIGEN:", ORIGEN)
print("Directorio de trabajo:", ROOT)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.colors import TwoSlopeNorm
from matplotlib.patches import Patch
from itertools import combinations
plt.style.use('dark_background')
COL = {'cyan':'#52dcf5','rosa':'#ed78db','naranja':'#ffa657','azul':'#8ea6ff','verde':'#86e0a3'}
orders=build_order_level(TABLAS)
items=build_item_level(TABLAS)
EXPORTAR=False  # Cambie a True para guardar figuras en figures/.
FIGDIR=ROOT/'figures'
if EXPORTAR: FIGDIR.mkdir(parents=True,exist_ok=True)
def mostrar(fig,nombre):
    fig.tight_layout()
    if EXPORTAR: fig.savefig(FIGDIR/f'{nombre}.png',dpi=160,bbox_inches='tight')
    plt.show(); plt.close(fig)
print('Origen:', ORIGEN, '| pedidos',len(orders),'| ítems',len(items))

### Galería: cómo leer cada gráfica

Esta galería compara **más de 25 representaciones** sobre las mismas tablas. Cada bloque contiene título, objetivo, código y aspectos a vigilar; la lista no pretende que todos los gráficos sean igualmente recomendables. Las **barras e histogramas** corresponden a tareas diferentes; los **boxplots y violines** describen distribuciones; los **mapas de calor** requieren escalas adecuadas y los **gráficos temporales** requieren verificar cobertura.

**Regla de interpretación:** describir patrón y denominador antes de sugerir causas. Los importes son subtotales de ítems o pagos según se indique: no son automáticamente utilidad o facturación neta.


## A. Variables categóricas
Las barras usan **longitud sobre un eje común**: normalmente facilitan comparaciones más precisas que áreas y ángulos. ¿Queremos conteos o proporciones? Hay que explicitar el denominador. Las barras no son histogramas: representan categorías separadas, no intervalos continuos.

In [ ]:
# Gráfico 1: barras verticales de estado del pedido
counts=orders['order_status'].value_counts()
fig,ax=plt.subplots(figsize=(9,4)); ax.bar(counts.index,counts.values,color=COL['cyan'])
ax.set(title='Pedidos por estado',ylabel='Número de pedidos'); ax.tick_params(axis='x',rotation=25)
mostrar(fig,'01_barras_estado')

In [ ]:
# Gráfico 2: barras horizontales ordenadas de estados geográficos (top 9)
counts=orders['customer_state'].value_counts().head(9).sort_values()
fig,ax=plt.subplots(figsize=(9,4)); ax.barh(counts.index,counts.values,color=COL['azul'])
ax.set(title='Pedidos por estado geográfico',xlabel='Número de pedidos')
mostrar(fig,'02_barras_horizontales')

In [ ]:
# Gráfico 3: barras agrupadas (mismas categorías, distinto estatus)
top=orders['customer_state'].value_counts().head(5).index
ct=pd.crosstab(orders.loc[orders.customer_state.isin(top),'customer_state'],
               orders.loc[orders.customer_state.isin(top),'order_status'])
keep=[k for k in ['delivered','shipped','canceled'] if k in ct]
fig,ax=plt.subplots(figsize=(9,4)); ct[keep].plot.bar(ax=ax,color=[COL['cyan'],COL['naranja'],COL['rosa']][:len(keep)])
ax.set(title='Estados de pedido: comparación agrupada',ylabel='Número de pedidos',xlabel='Estado geográfico'); ax.legend(title='Estatus')
mostrar(fig,'03_barras_agrupadas')

In [ ]:
# Gráfico 4: barras apiladas al 100%: composición, no volúmenes absolutos
prop=ct.div(ct.sum(axis=1),axis=0)[keep].fillna(0)
fig,ax=plt.subplots(figsize=(9,4)); prop.plot.bar(stacked=True,ax=ax,color=[COL['cyan'],COL['naranja'],COL['rosa']][:len(keep)])
ax.set(title='Composición de estatus (100%)',ylabel='Proporción de pedidos',xlabel='Estado'); ax.set_ylim(0,1)
ax.legend(title='Estatus'); mostrar(fig,'04_apiladas_100')

In [ ]:
# Gráfico 5: lollipop / puntos sobre eje (alternativa a barras)
vals=orders['medio_pago_inicial'].value_counts().head(6).sort_values()
fig,ax=plt.subplots(figsize=(9,4)); ax.hlines(vals.index,0,vals.values,color=COL['azul'],linewidth=2)
ax.plot(vals.values,vals.index,'o',color=COL['naranja'],markersize=9)
ax.set(title='Tipo de pago inicial (gráfico de puntos)',xlabel='Pedidos')
mostrar(fig,'05_lollipop')

**Práctica crítica:** una gráfica de pastel puede servir con pocas categorías, pero comparar ángulos es menos preciso que comparar longitudes. Usarla no debe ser un reflejo automático. Las gráficas ordinales conservan el orden lógico de sus niveles.

In [ ]:
# Gráfico 6: pastel, únicamente como contraste crítico de diseño
vals=orders['medio_pago_inicial'].value_counts().head(4)
fig,ax=plt.subplots(figsize=(6,5)); ax.pie(vals.values, labels=vals.index,autopct='%1.0f%%',
 colors=[COL['cyan'],COL['rosa'],COL['naranja'],COL['azul']][:len(vals)])
ax.set_title('Métodos de pago: lectura menos precisa de porcentajes')
mostrar(fig,'06_pastel_critica')

## B. Distribuciones: cantidad de intervalos y forma
En una variable continua, la anchura de los intervalos cambia la apariencia del histograma. En densidades y ECDF se comparan patrones de manera complementaria. **No presentaremos densidades como si fueran probabilidades de un único valor exacto.**

In [ ]:
# Gráficos 7 y 8: histograma con distintas decisiones de bins, MISMA muestra
serie=items['price'].dropna(); serie=serie.loc[serie.ge(0)]
tope=serie.quantile(.99) if len(serie)>30 else serie.max()
base=serie.loc[serie.le(tope)] # recorte de VISTA, no limpieza de datos
for n_bins in (10,35):
    fig,ax=plt.subplots(figsize=(8,4)); ax.hist(base,bins=n_bins,color=COL['cyan'],edgecolor='#101820')
    ax.set(title=f'Precio de ítem ≤ P99: {n_bins} intervalos',xlabel='Precio (BRL), vista truncada en P99',ylabel='Frecuencia')
    mostrar(fig,f'0{7 if n_bins==10 else 8}_hist_{n_bins}')
print('El recorte P99 solo afecta el GRÁFICO; la tabla original conserva los valores extremos.')

In [ ]:
# Gráfico 9: histograma de DENSIDAD + curva KDE (extensión opcional)
from scipy.stats import gaussian_kde
x=base.to_numpy(dtype=float)
fig,ax=plt.subplots(figsize=(8,4)); ax.hist(x,bins=25,density=True,color=COL['azul'],alpha=.55,label='Histograma (densidad)')
if len(np.unique(x))>1:
    grid=np.linspace(x.min(),x.max(),250); kde=gaussian_kde(x)
    ax.plot(grid,kde(grid),color=COL['naranja'],lw=2,label='KDE')
ax.set(title='Frecuencia relativa por unidad de precio',xlabel='Precio (BRL)',ylabel='Densidad')
ax.legend(); mostrar(fig,'09_densidad_kde')

In [ ]:
# Gráfico 10: ECDF (función de distribución empírica)
x=np.sort(serie.to_numpy()); y=np.arange(1,len(x)+1)/len(x)
fig,ax=plt.subplots(figsize=(8,4)); ax.step(x,y,where='post',color=COL['cyan'])
ax.set(title='Porcentaje de líneas con precio ≤ x',xlabel='Precio de ítem (BRL)',ylabel='Fracción acumulada')
ax.set_ylim(0,1); ax.set_xscale('symlog',linthresh=10)
mostrar(fig,'10_ecdf')

In [ ]:
# Gráfico 11: caja por grupos. Nota: bigotes no son intervalos de confianza.
estados=orders['customer_state'].value_counts().head(5).index
muestras=[orders.loc[orders.customer_state.eq(e),'subtotal_productos'].dropna().to_numpy() for e in estados]
fig,ax=plt.subplots(figsize=(9,4)); bp=ax.boxplot(muestras,tick_labels=list(estados),patch_artist=True,showfliers=False)
for p in bp['boxes']:p.set_facecolor(COL['azul'])
ax.set(title='Valores de pedido: cajas, sin dibujar los puntos extremos',ylabel='Subtotal de productos (BRL)')
mostrar(fig,'11_boxplot')

In [ ]:
# Gráfico 12: violín (extensión, resume una estimación de densidad)
fig,ax=plt.subplots(figsize=(9,4)); v=ax.violinplot(muestras,showmeans=False,showmedians=True)
for body in v['bodies']: body.set_facecolor(COL['rosa']);body.set_alpha(.65)
ax.set_xticks(range(1,len(estados)+1), list(estados));ax.set(title='Distribución completa estimada por estado',ylabel='Subtotal (BRL)')
ax.set_ylim(bottom=0,top=float(np.nanquantile(np.concatenate(muestras),.98))*2)
mostrar(fig,'12_violin')

In [ ]:
# Gráfico 13: boxplot horizontal + puntos de distribución con jitter
s=orders.loc[orders.order_status.eq('delivered'),'dias_hasta_entrega'].dropna()
s=s[s.ge(0)]
fig,ax=plt.subplots(figsize=(9,3));ax.boxplot(s,vert=False,positions=[0],showfliers=False)
vis=s.sample(min(600,len(s)),random_state=42)
ax.scatter(vis,np.random.default_rng(42).normal(0,.07,size=len(vis)),s=10,alpha=.22,color=COL['naranja'])
ax.set(title='Entrega: resumen y observaciones individuales',xlabel='Días desde la compra',yticks=[])
mostrar(fig,'13_box_jitter')

## C. Relaciones entre variables y superposición
Un gráfico de dispersión permite revisar forma, dirección, heterogeneidad y extremos antes de calcular correlaciones. En conjuntos grandes conviene aplicar transparencia, muestreo reproducible o celdas de densidad. **Correlación no implica causalidad.**

In [ ]:
# Gráfico 14: dispersión con transparencias (cada punto = un pedido)
sub=orders[['subtotal_productos','dias_hasta_entrega','customer_state']].dropna()
sub=sub.loc[sub.dias_hasta_entrega.ge(0)].sample(n=min(3500,len(sub)),random_state=2026)
fig,ax=plt.subplots(figsize=(9,4));ax.scatter(sub.subtotal_productos,sub.dias_hasta_entrega,alpha=.26,s=15,color=COL['cyan'])
ax.set(title='Precio de productos y tiempo de entrega',xlabel='Subtotal de productos (BRL)',ylabel='Días')
ax.set_xscale('symlog',linthresh=10);mostrar(fig,'14_dispersion_alpha')

In [ ]:
# Gráfico 15: dispersión coloreada por grupo (solo tres estados para legibilidad)
subset=sub[sub.customer_state.isin(sub.customer_state.value_counts().head(3).index)]
fig,ax=plt.subplots(figsize=(9,4))
for estado,grp in subset.groupby('customer_state'):
    ax.scatter(grp.subtotal_productos,grp.dias_hasta_entrega,s=14,alpha=.35,label=estado)
ax.set(title='Dispersión estratificada',xlabel='Subtotal (BRL)',ylabel='Días');ax.set_xscale('symlog',linthresh=10)
ax.legend(title='Estado');mostrar(fig,'15_dispersion_color')

In [ ]:
# Gráfico 16: hexbin (concentración) cuando los puntos se superponen
fig,ax=plt.subplots(figsize=(9,4));hb=ax.hexbin(sub.subtotal_productos,sub.dias_hasta_entrega,
 gridsize=28,mincnt=1,cmap='plasma',xscale='log' if sub.subtotal_productos.gt(0).all() else 'linear')
fig.colorbar(hb,ax=ax,label='Pedidos por celda');ax.set(title='Concentración de observaciones',xlabel='Subtotal (BRL)',ylabel='Días')
mostrar(fig,'16_hexbin')

In [ ]:
# Gráfico 17: histograma 2D / densidad por rejilla
fig,ax=plt.subplots(figsize=(9,4)); hb=ax.hist2d(sub.subtotal_productos.clip(upper=sub.subtotal_productos.quantile(.99)),
 sub.dias_hasta_entrega,bins=30,cmap='viridis')
fig.colorbar(hb[3],ax=ax,label='Pedidos');ax.set(title='Frecuencias por celdas 2D',xlabel='Subtotal acotado a P99 para VISTA (BRL)',ylabel='Días')
mostrar(fig,'17_hist_2d')

In [ ]:
# Gráfico 18: jitter para una variable ordinal (review_score)
qq=orders[['puntuacion_resena','dias_retraso']].dropna().copy();qq=qq[qq.dias_retraso.ge(-60)&qq.dias_retraso.le(60)]
qq=qq.sample(min(3500,len(qq)),random_state=42)
x_j=qq.puntuacion_resena.to_numpy(dtype=float)+np.random.default_rng(42).uniform(-.17,.17,len(qq))
fig,ax=plt.subplots(figsize=(9,4));ax.scatter(x_j,qq.dias_retraso,s=14,alpha=.25,color=COL['rosa'])
ax.axhline(0,color=COL['naranja'],ls='--');ax.set(xticks=[1,2,3,4,5],title='Puntuación ordinal vs retraso',xlabel='Puntuación de reseña',ylabel='Días de retraso')
mostrar(fig,'18_jitter_ordinal')

In [ ]:
# Gráfico 19: pequeños múltiplos / facetas manuales
fig,axes=plt.subplots(1,3,figsize=(12,3.5),sharex=True,sharey=True)
for ax,(estado,grp) in zip(axes,subset.groupby('customer_state')):
    ax.scatter(grp.subtotal_productos,grp.dias_hasta_entrega,s=13,alpha=.35,color=COL['cyan'])
    ax.set_title(estado);ax.set_xlabel('Subtotal BRL')
    ax.set_xscale('symlog',linthresh=10)
axes[0].set_ylabel('Días');fig.suptitle('Facetas por estado: mismas escalas para comparar')
mostrar(fig,'19_facetas')

## D. Series temporales y agregados
La forma de un gráfico temporal debe señalar períodos incompletos, huecos y variaciones de cobertura. Una curva de ingresos no prueba causalidad. La comparación requiere definir si se incluyen pedidos cancelados; aquí se muestra explícitamente el universo utilizado.

In [ ]:
monthly=(orders.dropna(subset=['mes_compra']).groupby('mes_compra')
         .agg(pedidos=('order_id','nunique'),valor=('subtotal_productos','sum'))
         .sort_index())
# Gráfico 20: línea temporal
fig,ax=plt.subplots(figsize=(10,4));ax.plot(range(len(monthly)),monthly.pedidos,'o-',color=COL['cyan'])
step=max(1,len(monthly)//10);ax.set_xticks(range(0,len(monthly),step),monthly.index[::step],rotation=40)
ax.set(title='Pedidos registrados por mes (cobertura observada)',ylabel='Pedidos')
mostrar(fig,'20_linea_mensual')

In [ ]:
# Gráfico 21: área de valor de productos por mes (NO equivale a facturación neta)
fig,ax=plt.subplots(figsize=(10,4));xx=np.arange(len(monthly));ax.fill_between(xx,monthly.valor,color=COL['naranja'],alpha=.35);ax.plot(xx,monthly.valor,color=COL['naranja'])
step=max(1,len(monthly)//10);ax.set_xticks(xx[::step],monthly.index[::step],rotation=40)
ax.set(title='Subtotal registrado de productos por mes',ylabel='BRL, excluye flete')
mostrar(fig,'21_area_mensual')

In [ ]:
# Gráfico 22: acumulado: el crecimiento siempre es monotónico por construcción
fig,ax=plt.subplots(figsize=(10,4));ax.step(np.arange(len(monthly)),monthly.pedidos.cumsum(),where='post',color=COL['verde'])
step=max(1,len(monthly)//10);ax.set_xticks(np.arange(0,len(monthly),step),monthly.index[::step],rotation=40)
ax.set(title='Acumulado de pedidos observados',ylabel='Pedidos acumulados')
mostrar(fig,'22_acumulado')

## E. Mapas de calor, correlaciones, valores faltantes y matrices
Para correlaciones usamos paleta **divergente centrada en cero** y escala fija de -1 a 1. En un mapa de faltantes, el color indica ausencia/presencia, no magnitud económica. Las coordenadas geográficas y prefijos postales tienen granularidad propia y no deben unirse sin agregación.

In [ ]:
# Gráfico 23: mapa de calor de correlaciones (valores numéricos auditables)
num=['subtotal_productos','total_flete','n_items','dias_hasta_entrega','dias_retraso','puntuacion_resena']
mat=orders[num].corr(method='pearson')
fig,ax=plt.subplots(figsize=(8,6));im=ax.imshow(mat,vmin=-1,vmax=1,cmap='coolwarm')
ax.set_xticks(range(len(num)),num,rotation=45,ha='right');ax.set_yticks(range(len(num)),num)
for i in range(len(num)):
 for j in range(len(num)):ax.text(j,i,f'{mat.iloc[i,j]:.2f}',ha='center',va='center',fontsize=8,color='black' if abs(mat.iloc[i,j])<.35 else 'white')
fig.colorbar(im,ax=ax,label='Correlación de Pearson');ax.set_title('Matriz de asociaciones (no causalidad)')
mostrar(fig,'23_matriz_correlacion')

In [ ]:
# Gráfico 24: patrón de faltantes en muestra reproducible
cols=['order_status','order_approved_at','order_delivered_customer_date','puntuacion_resena','total_pagado']
sample=orders[cols].sample(n=min(160,len(orders)),random_state=42)
fig,ax=plt.subplots(figsize=(10,4));im=ax.imshow(sample.isna().T,cmap='gray_r',aspect='auto',interpolation='nearest')
ax.set_yticks(range(len(cols)),cols);ax.set(xlabel='Pedidos muestreados',title='Blanco = faltante; oscuro = observado')
mostrar(fig,'24_faltantes')

In [ ]:
# Gráfico 25: matriz de dispersión pequeña, sin paquetes extra
cols=['subtotal_productos','total_flete','dias_hasta_entrega']
small=orders[cols].dropna().sample(n=min(500,len(orders[cols].dropna())),random_state=2026)
fig,axes=plt.subplots(3,3,figsize=(9,8))
for i,ycol in enumerate(cols):
 for j,xcol in enumerate(cols):
    ax=axes[i,j]
    if i==j:ax.hist(small[xcol],bins=20,color=COL['cyan'])
    else:ax.scatter(small[xcol],small[ycol],s=7,alpha=.22,color=COL['rosa'])
    if i==2:ax.set_xlabel(xcol,fontsize=8)
    if j==0:ax.set_ylabel(ycol,fontsize=8)
    ax.tick_params(labelsize=7)
fig.suptitle('Matriz de dispersión: lectura multivariada')
mostrar(fig,'25_scatter_matrix')

In [ ]:
# Gráfico 26: diagrama Q–Q como diagnóstico de normalidad, no prueba concluyente
from scipy import stats
vals=np.log1p(serie[serie.ge(0)])
fig,ax=plt.subplots(figsize=(7,4));stats.probplot(vals,dist='norm',plot=ax)
ax.set(title='Q–Q de log(1 + precio): desviaciones de normalidad',xlabel='Cuantiles teóricos',ylabel='Cuantiles observados')
mostrar(fig,'26_qq')

In [ ]:
# Gráfico 27: Pareto exploratorio, interpretando cuidadosamente la escala
s=items.groupby('categoria_mostrada')['price'].sum(min_count=1).dropna().sort_values(ascending=False).head(10)
fig,ax=plt.subplots(figsize=(10,4));ax.bar(np.arange(len(s)),s.values,color=COL['azul'])
ax.set_xticks(np.arange(len(s)),s.index,rotation=48,ha='right');ax.set_ylabel('BRL de productos')
ax.set_title('Principales categorías por subtotal de ítems (top 10)')
mostrar(fig,'27_pareto_barras')

## F. Opcional: visualización espacial de puntos agregados
La tabla `geolocation` contiene **múltiples coordenadas por prefijo postal**. Unir directamente con clientes produciría multiplicaciones ilegítimas. La siguiente figura muestra puntos sobre longitud/latitud, **no un mapa cartográfico** ni una estimación ponderada de ventas.

In [ ]:
from src.olist_tools import FILE_NAMES, detect_source
folder, _ = detect_source(ROOT)
geo_file=folder/FILE_NAMES['geolocation']
if geo_file.exists():
    geo=pd.read_csv(geo_file,usecols=['geolocation_zip_code_prefix','geolocation_lat','geolocation_lng'],
                    dtype={'geolocation_zip_code_prefix':'string'})
    resumen_geo=geo.groupby('geolocation_zip_code_prefix')[['geolocation_lat','geolocation_lng']].median().dropna()
    subset_geo=resumen_geo.sample(n=min(3500,len(resumen_geo)),random_state=42)
    fig,ax=plt.subplots(figsize=(8,4));ax.scatter(subset_geo.geolocation_lng,subset_geo.geolocation_lat,s=4,alpha=.25,color=COL['cyan'])
    ax.set(title='Coordenadas agregadas por prefijo postal (no mapa)',xlabel='Longitud',ylabel='Latitud')
    mostrar(fig,'28_latlon_agregadas')
else:
    print('Sin tabla geográfica: gráfico opcional omitido.')

## G. Variantes adicionales: mosaico, matriz temporal y puntos de intervalo

Estas tres visualizaciones amplían la caja de herramientas. El **mosaico** ilustra proporciones condicionales de categorías; el **mapa de calor temporal** relaciona día y hora, y el **intervalo de cuartiles con un punto** comunica mediana y dispersión sin sugerir incertidumbre inferencial.

In [ ]:
# Gráfico 29: mosaico condicional por estado del pedido (sin librerías extra)
from matplotlib.patches import Rectangle, Patch
cruce = pd.crosstab(orders["order_status"], orders["medio_pago_inicial"])
filas = cruce.sum(axis=1).sort_values(ascending=False).head(5).index
columnas = cruce.sum().sort_values(ascending=False).head(4).index
cruce = cruce.loc[filas, columnas]
fig, ax = plt.subplots(figsize=(10, 4))
xx = 0.0
for estado, r in cruce.iterrows():
    anchura = r.sum() / cruce.to_numpy().sum()
    yy = 0.0
    for j, (medio, cantidad) in enumerate(r.items()):
        alto = cantidad / r.sum() if r.sum() else 0
        ax.add_patch(Rectangle((xx, yy), anchura, alto, facecolor=list(COL.values())[j % len(COL)], edgecolor="white", alpha=0.75))
        yy += alto
    ax.text(xx + anchura / 2, -0.05, estado, ha="center", va="top", rotation=30, fontsize=8)
    xx += anchura
ax.set(xlim=(0, 1), ylim=(-0.17, 1), yticks=[0, 0.5, 1], ylabel="Proporción condicional")
ax.set_xticks([]); ax.set_title("Mosaico: estados × medio inicial de pago (selección top)")
leyenda = [Patch(facecolor=list(COL.values())[j], label=str(m)) for j, m in enumerate(columnas)]
ax.legend(handles=leyenda, loc="upper center", bbox_to_anchor=(0.5, 1), ncol=len(columnas), fontsize=8)
# La anchura de cada columna representa su peso en los pedidos seleccionados;
# su distribución vertical indica la mezcla de pagos condicional al estado.
mostrar(fig, "29_mosaico_categorias")

In [ ]:
# Gráfico 30: mapa de calor de actividad por día de semana y hora (compra)
compras = TABLAS["orders"]["order_purchase_timestamp"].dropna()
frecuencia = pd.crosstab(compras.dt.dayofweek, compras.dt.hour).reindex(index=range(7), columns=range(24), fill_value=0)
fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(frecuencia, cmap="viridis", aspect="auto")
ax.set_yticks(range(7), ["Lun", "Mar", "Mié", "Jue", "Vie", "Sáb", "Dom"])
ax.set_xticks(range(0, 24, 2));ax.set(xlabel="Hora de la compra", title="Registros de compra por día y hora")
fig.colorbar(im, ax=ax, label="Pedidos")
mostrar(fig, "30_heatmap_dia_hora")

In [ ]:
# Gráfico 31: intervalos descriptivos Q1–Q3 y mediana por región (NO IC)
estados = orders["customer_state"].value_counts().head(8).index
r = orders.loc[orders["customer_state"].isin(estados)].groupby("customer_state")["subtotal_productos"].quantile([.25,.5,.75]).unstack()
r = r.loc[estados]
fig, ax = plt.subplots(figsize=(9, 4))
for i, (est, q) in enumerate(r.iterrows()):
    ax.hlines(i, q[.25], q[.75], color=COL["azul"], linewidth=5)
    ax.plot(q[.5], i, "o", color=COL["naranja"], markersize=8)
ax.set_yticks(range(len(r)), r.index); ax.invert_yaxis()
ax.set(xlabel="Subtotal (BRL)", title="Punto = mediana; segmento = rango intercuartílico")
mostrar(fig, "31_intervalos_cuartiles")

## H. Diagnóstico y exportación
**Diseño responsable:** evita 3D decorativo, ejes truncados en barras sin advertencia, paletas incongruentes y comparaciones de tamaños diferentes sin denominador. Considera legibilidad en proyección, accesibilidad cromática y tipo de audiencia. `EXPORTAR=True` guarda las imágenes en `figures/` sin escribir sobre los datos.

**Actividades:**
- **Esencial:** selecciona seis gráficas distintas, escribe una interpretación y una limitación por gráfica.
- **Interpretación:** compara un pastel y barras; explica cuál revela mejor diferencias pequeñas. Contrasta histogramas con 10 y 35 intervalos.
- **Extensión:** rediseña una gráfica engañosa; implementa `hexbin` o KDE con un parámetro de suavizado distinto y evalúa su efecto.

**Lecturas:** notas de visualización anteriores (Gestalt, color, densidad, cajas, violines, dispersión, mosaicos); [Matplotlib](https://matplotlib.org/stable/users/index.html), [Fundamentals of Data Visualization](https://clauswilke.com/dataviz/).

In [ ]:
# Opcional: activar EXPORTAR=True en la celda de configuración y volver a ejecutar
# los gráficos deseados. Su interpretación escrita va aquí.